# Step 3: Machine Learning Model Training and Cross-Validation

## Overview
Trains prediction models using the prepared clinical features to forecast neutropenic events.

### Validation Pipeline
1. **Patient-level Data Splitting**: Stratified Group 5-Fold Cross-Validation grouped by patient ID (`hs`). This prevents data leakage (ensuring the same patient does not overlap between train and test splits) while maintaining class distribution.
2. **Missing Value Imputation**: Continuous clinical lab values and somatometry indicators are filled using the **median values** derived exclusively from the training partition.
3. **Feature Scaling**: Z-score normalization (`StandardScaler`) is applied to numerical features.
4. **Class Imbalance Mitigation**: Computes class weights adjusted via the square root of the imbalance ratio for XGBoost and LightGBM:
   $$\text{scale\_pos\_weight} = \sqrt{\frac{N_{\text{negative}}}{N_{\text{positive}}}}$$

### Algorithms Trained
- **Random Forest Classifier**
- **XGBoost Classifier**
- **LightGBM Classifier**

## Inputs and Outputs
- **Input**: `../../data/step2_processing_outputs/step2_all_tagt_feat`
- **Output**: Trained model parameters (`../../data/step3_model_outputs/all_*_model.joblib`) and normalized test sets (`../../data/step3_model_outputs/step3_all_*`)



In [ ]:
import pandas as pd
import numpy as np 
import os
import joblib
import lightgbm as lgb

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler # Feature scaling
from sklearn.ensemble import RandomForestClassifier 
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [ ]:
data_dir = '../../data'
input_dir = f'{data_dir}/step2_processing_outputs' 
output_dir = f'{data_dir}/step3_model_outputs'

file_paths = {
    'tagt_feat': f'{input_dir}/step2_all_tagt_feat'
}


In [ ]:
# Dict to store loaded DataFrames
dataframes = {}

# Read Parquet files for defined file paths
for key, file_path in file_paths.items():
    if not os.path.exists(file_path):
        print(f"Warning: ファイル '{file_path}' が存在しません。スキップします。")
        continue
        
    try:
        # lineterminator='\n' を指定
        df = pd.read_parquet(file_path)
        dataframes[key] = df
        print(f"'{key}'loaded successfully. Shape: {df.shape}")
    except Exception as e:
        print(f"Error: '{file_path}' failed to load. Error: {e}")

In [ ]:
# Load processed data
df_tagt_feat = dataframes['tagt_feat']

# Extract feature column names
feat_column_names = df_tagt_feat.drop(columns=['hs', 'Event', 'DO_DATE']).columns.tolist()

# Define feature matrix (X) and target vector (y)
features = feat_column_names
target = 'Event'

# Assign patient ID (hs) as the DataFrame index
X = df_tagt_feat[features].copy()
X.index = df_tagt_feat['hs']  # インデックスに hs をセット

y = df_tagt_feat[target].copy()
y.index = df_tagt_feat['hs']  # Assign hs as index for y

# Split data using StratifiedGroupKFold
from sklearn.model_selection import StratifiedGroupKFold
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

# Set aside 20% of patients for testing
groups = df_tagt_feat['hs']
train_idx, test_idx = next(sgkf.split(X, y, groups=groups))

# Index-based division into train and test sets
X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()

# Imputation of missing values
fill_cols = ['lastNEUT#', 'lastWBC','MS_HEIGHT', 'MS_WEIGHT']

# Impute using median values computed from the training set
train_medians = X_train[fill_cols].median()
X_train[fill_cols] = X_train[fill_cols].fillna(train_medians)
X_test[fill_cols] = X_test[fill_cols].fillna(train_medians)

# Scale features
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

# Reconstruct scaled DataFrames keeping patient index
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=features, index=X_train.index)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=features, index=X_test.index)

# Convert target arrays to DataFrames with patient index
y_train_df = pd.DataFrame(y_train)
y_train_df.columns = ['Event']
y_test_df = pd.DataFrame(y_test)
y_test_df.columns = ['Event']

# Export scaled test data for model evaluation
X_test_scaled_df.to_parquet(f'{output_dir}/step3_all_x_test_scaled')
X_test_scaled_df.to_excel(f'{output_dir}/step3_all_x_test_scaled.xlsx', sheet_name='x_test_scaled')
y_test_df.to_parquet(f'{output_dir}/step3_all_y_test')
y_test_df.to_excel(f'{output_dir}/step3_all_y_test.xlsx', sheet_name='y_test')

In [ ]:
# Model Selection and Training
# 1. Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=20,min_samples_leaf=10)
rf_model.fit(X_train_scaled_df, y_train) 

In [ ]:
# 2. XGBoost Classifier

# 2.1 Calculate class imbalance ratio
# Ratio of negative to positive class samples
count_neg = (y_train_df == 0).sum().iloc[0]
count_pos = (y_train_df == 1).sum().iloc[0]

ratio = count_neg / count_pos

# 2.2 Compute class weights for imbalance handling
tuned_weight = np.sqrt(ratio)

print(f"Negative samples: {count_neg}")
print(f"Positive samples: {count_pos}")
print(f"Calculated ratio: {ratio:.2f}")
print(f"Tuned weight (scale_pos_weight): {tuned_weight:.2f}")

# Recalculate weights
tuned_weight = np.sqrt(ratio)

xgb_model = XGBClassifier(
    scale_pos_weight=tuned_weight,  
    n_estimators=1000,
    learning_rate=0.05, 
    max_depth=7, 
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='aucpr',
    early_stopping_rounds=200
)

# Fit XGBoost model using test set for early stopping
xgb_model.fit(
    X_train_scaled, 
    y_train, 
    eval_set=[(X_test_scaled, y_test)], 
    verbose=False
)

In [ ]:
# Recalculate weights
tuned_weight = np.sqrt(ratio)

# 3. LightGBM Classifier
lgb_model = LGBMClassifier(
    scale_pos_weight=tuned_weight,
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=7,
    num_leaves=31,              
    min_child_samples=5,       
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    importance_type='gain',     
    verbosity=-1                
)

# Fit LightGBM model with early stopping
lgb_model.fit(
    X_train_scaled, 
    y_train,
    eval_set=[(X_test_scaled, y_test)],
    eval_metric='auc',          
    callbacks=[
        lgb.early_stopping(stopping_rounds=200),
        lgb.log_evaluation(period=0) # Suppress training progress log
    ]
)

In [ ]:
# Save trained models
joblib.dump(rf_model,  f'{output_dir}/all_rf_model.joblib')
joblib.dump(xgb_model,  f'{output_dir}/all_xgb_model.joblib')
joblib.dump(lgb_model,  f'{output_dir}/all_lgb_model.joblib')

In [ ]:
print(X_train.isnull().sum().sort_values(ascending=False))